In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, List
from collections import Counter
import logging, os

load_dotenv()


True

In [14]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, List
from collections import Counter
import logging, os

class CoTResult(BaseModel):
    category: Literal['Billing','Technical','Feature Request','Churn Risk','Spam','Other']
    urgency: Literal['High', 'Medium', 'Low']
    confidence: int = Field(ge=1, le=10)
    reasoning: str = Field(description='One sentence explanation')
    cot_steps: List[str] = Field(description='3-5 reasoning steps')

# Schema for Tree of Thought branch
class ToTBranch(BaseModel):
    branch_name: str
    reasoning: str
    confidence: int = Field(ge=1, le=10)

# Schema for ToT result
class ToTResult(BaseModel):
    branches: List[ToTBranch] = Field(description='2-3 possible interpretations')
    selected_branch: str
    final_category: Literal['Billing','Technical','Feature Request','Churn Risk','Spam','Other']
    final_urgency: Literal['High', 'Medium', 'Low']
    final_reasoning: str



FALLBACK = CoTResult(
    category='Other', urgency='Medium', confidence=1,
    reasoning='Parse failed — routed to general support',
    cot_steps=['Parse failed']
)

In [5]:
cot_parser = PydanticOutputParser(pydantic_object=CoTResult)

In [6]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, max_tokens=800)


In [7]:

COT_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '''Expert support email classifier.

Rules:
- Login broken after payment → Billing (NOT Technical)
- App crashes → Technical
- Pricing + evaluating alternatives → Churn Risk
- Feature requests → Feature Request
- Spam → Spam

Think step by step before classifying.

{format_instructions}'''),
    ('human', 'Subject: {subject}\nBody: {body}')
]).partial(format_instructions=cot_parser.get_format_instructions())

cot_chain = COT_PROMPT | llm | cot_parser


In [9]:
email = {
    'subject': "Can't login — paid for annual plan last week",
    'body': 'Our entire team cannot login. We paid for the annual plan '
            'last week. We have a board demo in 3 hours!'
}

cot_chain.invoke(email)

CoTResult(category='Billing', urgency='High', confidence=9, reasoning='The login issue is directly related to the payment made for the annual plan.', cot_steps=['The user mentions they cannot login.', 'They state they paid for the annual plan last week.', 'The issue is affecting the entire team.', 'There is an urgent board demo in 3 hours.', 'This indicates a billing-related issue rather than a technical one.'])

In [15]:
tot_parser = PydanticOutputParser(pydantic_object=ToTResult)

TOT_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '''Expert email analyzer using Tree of Thought.

Consider MULTIPLE interpretations before choosing.

Steps:
1. Generate 2-3 different ways to interpret this email
2. Explain reasoning for each
3. Select the best interpretation
4. Give final classification

{format_instructions}'''),
    ('human', 'Subject: {subject}\nBody: {body}')
]).partial(format_instructions=tot_parser.get_format_instructions())

tot_chain = TOT_PROMPT | llm | tot_parser



In [16]:
tot_chain.invoke(email)

ToTResult(branches=[ToTBranch(branch_name='Login Issue Due to Payment Processing', reasoning='The user mentions that they paid for the annual plan last week, which could imply that the payment has not been processed correctly, leading to login issues for the entire team.', confidence=8), ToTBranch(branch_name='Urgent Need for Support', reasoning='The email indicates a time-sensitive situation with a board demo in 3 hours, suggesting that the user is seeking immediate assistance to resolve the login issue before the demo.', confidence=9), ToTBranch(branch_name='Technical Glitch Affecting Multiple Users', reasoning='The mention of the entire team being unable to log in could indicate a broader technical issue affecting multiple users, rather than just a single account problem.', confidence=7)], selected_branch='Urgent Need for Support', final_category='Technical', final_urgency='High', final_reasoning='The email highlights an urgent login issue affecting the entire team, with a critical 

In [17]:
INJECTION_PHRASES = [
    'ignore all previous', 'ignore your instructions',
    'print your prompt', 'reveal system prompt',
    'you are now', 'forget everything'
]

def check_input(email: dict) -> tuple:
    """
    Checker component — scans email before sending to LLM.
    Returns: (is_safe: bool, reason: str)
    """
    subject = email.get('subject', '').strip()
    body    = email.get('body', '').strip()

    if not body and not subject:
        return False, 'Empty email'
    if len(body) > 5000:
        return False, f'Too long ({len(body)} chars)'

    combined = (subject + ' ' + body).lower()
    for phrase in INJECTION_PHRASES:
        if phrase in combined:
            return False, f'Injection detected: "{phrase}"'

    return True, 'Safe'



In [ ]:
def classify_with_cot(email):
    safe_chain = cot_chain.with_retry(
        stop_after_attempt=3,
        wait_exponential_jitter=True
    )

    try:
        result = safe_chain.invoke(email)
        return result
    except Exception as e:
        return FALLBACK


classify_with_cot(email)


In [12]:
SPAM_KEYWORDS = ["lottery", "winner", "click here", "free money", "urgent", "congratulations", "prize"]

def validate_email(email):
    # print("before parsing email: ",email['question'])
    body = email.get("question","").lower()
    # print("after parsing email: ",email['question'])
    found_spam = [word for word in SPAM_KEYWORDS if word in body]
    print("found_spam",found_spam)
    if found_spam:
        raise ValueError("Spam word found:",found_spam)
    return email

In [13]:
validate_email({"question": "Congratulations! You are a lottery winner"})

found_spam ['lottery', 'winner', 'congratulations']


ValueError: ('Spam word found:', ['lottery', 'winner', 'congratulations'])

In [ ]:
chain = prompt | validate_email | llm | parser 